# Ingest data from CSV and clean it


Check required columns and unexpected schema changes.



Handle missing values according to documented rules.



Detect and investigate duplicate records.



Validate data types, ranges and business rules.



Standardise text, dates and numeric formats.



Flag anomalies and quarantine invalid records.



Generate before and after quality metrics.


## Installing dependencies

In [99]:
%pip install pathlib
%pip install pandas
%pip install kagglehub
%pip install scipy
%pip install numpy
%pip install seaborn
%pip install matplotlib


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgr

## Loading datsaet from kagglehub

In [100]:
# import pandas
import pandas as pd

# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "transaction_data.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "vipin20/transaction-data",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)


/var/folders/wx/srl0047n7ggf6r_b5glkrctr0000gn/T/ipykernel_32385/3204598897.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


## Defining paths

In [101]:
from pathlib import Path

PROJECT_DIR = Path("/Users/hamzashah/data-platform-engineering-snowflake")

input_path = PROJECT_DIR / "datasets" / "raw" / "transaction_data.csv"
output_path = PROJECT_DIR / "datasets" / "cleaned" / "utf8_csv.csv"




## Convert into UTF-8 Encoding each time
Regardless of CSV encoding when the dataset is input, we want to convert it into one specific type of CSV encoding, so that moving forward, all our cleaned datasets have the same encoding.

The encoding i have chosen is: UTF-8

In [102]:
def convert_to_utf8(input_path, output_path):
    encodings = ["utf-8-sig", "utf-8", "cp1252"]

    for encoding in encodings:
        try:
            df = pd.read_csv(input_path, encoding=encoding)

            df.to_csv(
                output_path,
                index = False,
                encoding = "utf-8"
            )

            print("converted to UTF-8 Encoding, source encoding: {encoding}")
            return df

        except UnicodeDecodeError:
            continue
    
    raise ValueError("could not decode CSV with supported encodingsS")

In [103]:
df = convert_to_utf8(
    input_path,
    output_path
)

df = pd.read_csv(
    output_path,
    encoding="utf-8"
)

converted to UTF-8 Encoding, source encoding: {encoding}


## Inspecting the dataset

In [104]:
#inspecting the dataset

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

display(df.head())

df.info() #inspecting columns and datatypes

print(df.dtypes)

display(df.describe(include='all').T) # summary



,userid,transactionid,transactiontime,itemcode,itemdescription,numberofitemspurchased,costperitem,country
0,278166,6355745,Sat Feb 02 12:50:00 IST 2019,465549,FAMILY ALBUM WHITE PICTURE FRAME,6,11.73,United Kingdom
1,337701,6283376,Wed Dec 26 09:06:00 IST 2018,482370,LONDON BUS COFFEE MUG,3,3.52,United Kingdom
2,267099,6385599,Fri Feb 15 09:45:00 IST 2019,490728,SET 12 COLOUR PENCILS DOLLY GIRL,72,0.90,France
3,380478,6044973,Fri Jun 22 07:14:00 IST 2018,459186,UNION JACK FLAG LUGGAGE TAG,3,1.73,United Kingdom
4,-1,6143225,Mon Sep 10 11:58:00 IST 2018,1733592,WASHROOM METAL SIGN,3,3.40,United Kingdom


<class 'pandas.DataFrame'>
RangeIndex: 1083818 entries, 0 to 1083817
Data columns (total 8 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   userid                  1083818 non-null  int64  
 1   transactionid           1083818 non-null  int64  
 2   transactiontime         1083818 non-null  str    
 3   itemcode                1083818 non-null  int64  
 4   itemdescription         1080910 non-null  str    
 5   numberofitemspurchased  1083818 non-null  int64  
 6   costperitem             1083818 non-null  float64
 7   country                 1083818 non-null  str    
dtypes: float64(1), int64(4), str(3)
memory usage: 66.2 MB
userid                      int64
transactionid               int64
transactiontime               str
itemcode                    int64
itemdescription               str
numberofitemspurchased      int64
costperitem               float64
country                       str
dtype: object


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
userid,1083818.0,NaN,NaN,NaN,241016.218825,142336.431264,-1.0,259392.0,302022.0,341355.0,384027.0
transactionid,1083818.0,NaN,NaN,NaN,6159416.640572,147634.09388,5900015.0,6026856.0,6166611.0,6289569.0,6397457.0
transactiontime,1083818,23260,Sat Jan 12 10:41:00 IST 2019,2228,NaN,NaN,NaN,NaN,NaN,NaN,NaN
itemcode,1083818.0,NaN,NaN,NaN,658268.696312,452631.426503,-1.0,460908.0,475293.0,488943.0,1894494.0
itemdescription,1080910,4223,WHITE HANGING HEART T-LIGHT HOLDER,4738,NaN,NaN,NaN,NaN,NaN,NaN,NaN
numberofitemspurchased,1083818.0,NaN,NaN,NaN,28.656749,654.243172,-242985.0,3.0,9.0,30.0,242985.0
costperitem,1083818.0,NaN,NaN,NaN,9.498798,2308.138578,-15265.64,1.73,2.88,5.7,1696285.44
country,1083818,38,United Kingdom,990956,NaN,NaN,NaN,NaN,NaN,NaN,NaN


transaction time can be split up into transaction date, and transaction time

## Step 1: identify missing values

In [105]:
missing_values_df = pd.DataFrame({
    "Column": ["userid", "transactionid", "transactiontime", "itemcode", "itemdescription", "numberofitemspurchased", "costperitem", "country"],
    "Num missing": [df["userid"].isna().sum(),
    df["transactionid"].isna().sum(),
    df["transactiontime"].isna().sum(),
    df["itemcode"].isna().sum(),
    df["itemdescription"].isna().sum(),
    df["numberofitemspurchased"].isna().sum(),
    df["costperitem"].isna().sum(),
    df["country"].isna().sum(),
    ]
})

display(missing_values_df)

,Column,Num missing
0,userid,0
1,transactionid,0
2,transactiontime,0
3,itemcode,0
4,itemdescription,2908
5,numberofitemspurchased,0
6,costperitem,0
7,country,0


We can see from the above that we are missing Item Descriptions for many items, this needs to be investigated.

## Step 2: detect outliers in numerical columns

In [106]:
import numpy as np
from scipy.stats import zscore

# only look at columns which are numeric
numeric_cols = df.select_dtypes(include="number").columns.difference(["userid", "transactionid", "transactiontime", "itemcode"])

# check for statistical outliers using Z scores
outliers_df = pd.DataFrame({
    "Column": numeric_cols,
    "Outliers": [
        (np.abs(zscore(df[col].dropna())) > 3).sum()
        for col in numeric_cols
    ],
    # display skew
    "Skew": [df[col].skew()
    for col in numeric_cols]
    
})

display(outliers_df)

,Column,Outliers,Skew
0,costperitem,68,732.487315
1,numberofitemspurchased,692,-0.264076


As we can see, there are some statistical outliers in the cost per item and number of items purchased fields. This will need to be investigated later.

## Step 3: Duplicate rows

Identify the number of duplicate rows

In [107]:
original_cols = [
    "userid",
    "transactionid",
    "transactiontime",
    "itemcode",
    "itemdescription",
    "numberofitemspurchased",
    "costperitem",
    "country"
]

print("\nDuplicate rows before removal:")
print(df.duplicated().sum())

#remove duplicates
df = df.drop_duplicates(subset=original_cols).copy()

# Verify
print("Duplicate rows after removal:", df.duplicated(subset=original_cols).sum())
print("Rows remaining:", len(df))


Duplicate rows before removal:
547246
Duplicate rows after removal: 0
Rows remaining: 536572


## Step 4: Standardise formats

1 - Im going to split the Transaction time field (Sat Feb 02 12:50:00 IST 2019)

into transaction date (Feb 02 2019)

and transaction time (12:50:00 IST).

Im going to drop the day (Sat), because it is not relevant for our data reporting needs, and can be found using the date anyways if needed later.

2 -  The text fields can be standardised by removing leading and trailing whitespace

3 - The missing descriptions can be filled using a lookup on the same item code and filling it in


In [108]:
# convert timestamp into a datetime type
df["transactiontime"] = pd.to_datetime(
    df["transactiontime"],
    format = "%a %b %d %H:%M:%S IST %Y"
)

# chcek if anything did not parse
print("Invalid timestamps:", df["transactiontime"].isna().sum())

#split timestamp into date and time
df["transaction_date"] = df["transactiontime"].dt.date

df["transaction_clock_time"] = (
    df["transactiontime"].dt.time
)

Invalid timestamps: 0


In [109]:
# 2 - removing leading and trailing whitespace in text columnns
text_cols = ["itemdescription", "country"]

for col in text_cols:
    df[col] = df[col].str.strip()

In [110]:
# 3 - for items without a description, we can look up the same item and use that desription to fill it in

lookup = df.dropna(subset=["itemdescription"]).drop_duplicates("itemcode").set_index("itemcode")["itemdescription"]

descriptions = df["itemcode"].map(lookup)

df["itemdescription"] = df["itemdescription"].fillna(descriptions)


# now check how many descriptionsa re missing after this lookup and replace
print("number of descriptions missing:", df["itemdescription"].isna().sum())

number of descriptions missing: 91


there are 182 descriptions missing, which have no lookup available, we cant fill these ones in so will leave them blank.

### Next we should drop negative prices, user IDs = 1 or 0, and transaction IDs = 1 or 0

In [115]:
# Remove rows with invalid user ID, transaction IDs, or quanitity

rows_before = len(df)

print("Rows before cleaning:", rows_before)

df = df[
    (~df["userid"].isin([-1, 0])) &
    (~df["transactionid"].isin([-1, 0])) &
    (~df["itemcode"].isin([-1, 0])) &
    (df["costperitem"] >= 0) &
    (df["numberofitemspurchased"] >= 0)
].copy()

print("Rows removed:", rows_before - len(df))
print("Rows remaining:", len(df))

Rows before cleaning: 392698
Rows removed: 1416
Rows remaining: 391282
